In [1]:
!nvidia-smi
!pip install nibabel tqdm


Thu Feb  5 08:31:16 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   54C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os
import numpy as np
import nibabel as nib
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from google.colab import drive


In [4]:
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
#to get faster epochs , copying from g drive into local ssd folder of collab
!cp -r /content/drive/MyDrive/brats2021/BraTS2021_Training_Data /content/
BRATS_ROOT = "/content/BraTS2021_Training_Data"


In [6]:
# Change this to my Kaggle-extracted path
BRATS_ROOT = "/content/BraTS2021_Training_Data"


MODEL_SAVE_DIR = "/content/drive/MyDrive/brats_models"
os.makedirs(MODEL_SAVE_DIR, exist_ok=True)


In [7]:
import os

print("Exists:", os.path.exists(BRATS_ROOT))
print("Num patients:", len(os.listdir(BRATS_ROOT)))
print(os.listdir(BRATS_ROOT)[:5])


Exists: True
Num patients: 1251
['BraTS2021_00586', 'BraTS2021_00512', 'BraTS2021_01065', 'BraTS2021_00366', 'BraTS2021_00291']


In [8]:
def load_nifti(path):
    return nib.load(path).get_fdata()

def zscore_norm(volume):
    mask = volume > 0
    mean = volume[mask].mean()
    std = volume[mask].std()
    volume[mask] = (volume[mask] - mean) / (std + 1e-8)
    return volume


In [9]:
def center_crop(enc_feat, target_feat):
    """
    Crop enc_feat spatially to match target_feat size
    """
    _, _, d, h, w = target_feat.shape
    enc_d, enc_h, enc_w = enc_feat.shape[2:]

    d1 = (enc_d - d) // 2
    h1 = (enc_h - h) // 2
    w1 = (enc_w - w) // 2

    return enc_feat[:, :, d1:d1+d, h1:h1+h, w1:w1+w]


In [10]:
def random_crop_3d(image, label, crop_size=(128, 128, 128)):
    """
    image: (C, D, H, W)
    label: (D, H, W)
    """
    _, D, H, W = image.shape
    cd, ch, cw = crop_size

    d1 = np.random.randint(0, D - cd + 1)
    h1 = np.random.randint(0, H - ch + 1)
    w1 = np.random.randint(0, W - cw + 1)

    image_crop = image[:, d1:d1+cd, h1:h1+ch, w1:w1+cw]
    label_crop = label[d1:d1+cd, h1:h1+ch, w1:w1+cw]

    return image_crop, label_crop


In [11]:
class BraTSDataset(Dataset):
    def __init__(self, patient_dirs):
        self.patient_dirs = patient_dirs
        self.modalities = ["t1", "t1ce", "t2", "flair"]

    def __len__(self):
        return len(self.patient_dirs)

    def __getitem__(self, idx):
        pdir = self.patient_dirs[idx]
        pid = os.path.basename(pdir)

        images = []
        for m in self.modalities:
            vol = load_nifti(os.path.join(pdir, f"{pid}_{m}.nii.gz"))
            vol = zscore_norm(vol)
            images.append(vol)

        image = np.stack(images, axis=0)  # (C, D, H, W)

        seg = load_nifti(os.path.join(pdir, f"{pid}_seg.nii.gz"))
        seg[seg == 4] = 3

        # ---- PATCH SAMPLING ----
        image, seg = random_crop_3d(image, seg, crop_size=(128, 128, 128))

        return torch.tensor(image, dtype=torch.float32), torch.tensor(seg, dtype=torch.long)



In [12]:
all_patients = sorted([
    os.path.join(BRATS_ROOT, p)
    for p in os.listdir(BRATS_ROOT)
    if p.startswith("BraTS")
])

train_patients = all_patients[:50]
val_patients   = all_patients[50:60]


In [13]:
print(all_patients[0])
print(os.listdir(all_patients[0]))


/content/BraTS2021_Training_Data/BraTS2021_00002
['BraTS2021_00002_t1.nii.gz', 'BraTS2021_00002_t2.nii.gz', 'BraTS2021_00002_t1ce.nii.gz', 'BraTS2021_00002_seg.nii.gz', 'BraTS2021_00002_flair.nii.gz']


In [14]:
import os
print(os.path.exists("/content/BraTS2021_Training_Data"))
print(len(os.listdir("/content/BraTS2021_Training_Data")))


True
1251


In [15]:
print(train_patients[0])
print(os.listdir(train_patients[0]))


/content/BraTS2021_Training_Data/BraTS2021_00002
['BraTS2021_00002_t1.nii.gz', 'BraTS2021_00002_t2.nii.gz', 'BraTS2021_00002_t1ce.nii.gz', 'BraTS2021_00002_seg.nii.gz', 'BraTS2021_00002_flair.nii.gz']


In [16]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))


True
Tesla T4


In [17]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class DoubleConv3D(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv3d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.InstanceNorm3d(out_channels),
            nn.ReLU(inplace=True),

            nn.Conv3d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.InstanceNorm3d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class UNet3D(nn.Module):
    def __init__(self, in_channels=4, num_classes=4):
        super().__init__()

        # Encoder
        self.enc1 = DoubleConv3D(in_channels, 32)
        self.enc2 = DoubleConv3D(32, 64)
        self.enc3 = DoubleConv3D(64, 128)

        self.pool = nn.MaxPool3d(kernel_size=2)

        # Decoder
        self.up2 = nn.ConvTranspose3d(128, 64, kernel_size=2, stride=2)
        self.dec2 = DoubleConv3D(128, 64)

        self.up1 = nn.ConvTranspose3d(64, 32, kernel_size=2, stride=2)
        self.dec1 = DoubleConv3D(64, 32)

        self.out_conv = nn.Conv3d(32, num_classes, kernel_size=1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))

        d2 = self.up2(e3)
        e2_crop = center_crop(e2, d2)
        d2 = self.dec2(torch.cat([d2, e2_crop], dim=1))

        d1 = self.up1(d2)
        e1_crop = center_crop(e1, d1)
        d1 = self.dec1(torch.cat([d1, e1_crop], dim=1))

        return self.out_conv(d1)


In [18]:
model = UNet3D()
print(model)


UNet3D(
  (enc1): DoubleConv3D(
    (block): Sequential(
      (0): Conv3d(4, 32, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
      (1): InstanceNorm3d(32, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
      (2): ReLU(inplace=True)
      (3): Conv3d(32, 32, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
      (4): InstanceNorm3d(32, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
      (5): ReLU(inplace=True)
    )
  )
  (enc2): DoubleConv3D(
    (block): Sequential(
      (0): Conv3d(32, 64, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
      (1): InstanceNorm3d(64, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
      (2): ReLU(inplace=True)
      (3): Conv3d(64, 64, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
      (4): InstanceNorm3d(64, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
      (5): ReLU(inplace=True)
    )
  )
  (enc3): DoubleConv3D(
   

In [19]:
import torch.nn.functional as F
import torch.nn as nn

def dice_loss(pred, target, smooth=1e-5):
    """
    pred: (B, C, D, H, W) logits
    target: (B, D, H, W) labels
    """
    pred = F.softmax(pred, dim=1)
    loss = 0.0

    # skip background (class 0)
    for c in range(1, 4):
        p = pred[:, c]
        t = (target == c).float()

        intersection = (p * t).sum()
        union = p.sum() + t.sum()

        loss += 1 - (2.0 * intersection + smooth) / (union + smooth)

    return loss / 3.0


class DiceCELoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.ce = nn.CrossEntropyLoss()

    def forward(self, pred, target):
        return self.ce(pred, target) + dice_loss(pred, target)


In [20]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))


Using device: cuda
GPU: Tesla T4


In [21]:
from torch.utils.data import DataLoader

# ---- Train / Validation split ----
all_patients = sorted([
    os.path.join(BRATS_ROOT, p)
    for p in os.listdir(BRATS_ROOT)
    if p.startswith("BraTS")
])

train_patients = all_patients[:50]
val_patients   = all_patients[50:60]

print("Train patients:", len(train_patients))
print("Val patients  :", len(val_patients))

# ---- Dataset objects ----
train_ds = BraTSDataset(train_patients)
val_ds   = BraTSDataset(val_patients)

# ---- DataLoaders ----
train_loader = DataLoader(
    train_ds,
    batch_size=1,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_ds,
    batch_size=1,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)


Train patients: 50
Val patients  : 10


In [22]:
# ---- Sanity check (NO training yet) ----

model = UNet3D().to(device)
criterion = DiceCELoss()

model.train()

x, y = next(iter(train_loader))
x = x.to(device)
y = y.to(device)

print("Input shape :", x.shape)    # (1, 4, D, H, W)
print("Target shape:", y.shape)    # (1, D, H, W)

out = model(x)
print("Output shape:", out.shape)  # (1, 4, D, H, W)

loss = criterion(out, y)
print("Loss value:", loss.item())

# cleanup
del x, y, out, loss
torch.cuda.empty_cache()


Input shape : torch.Size([1, 4, 128, 128, 128])
Target shape: torch.Size([1, 128, 128, 128])
Output shape: torch.Size([1, 4, 128, 128, 128])
Loss value: 2.4855096340179443


In [23]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

EPOCHS = 5   # you can change to 10 later

for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss = 0.0

    for x, y in train_loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        # free memory aggressively (important for Colab)
        del out, loss
        torch.cuda.empty_cache()

    avg_loss = running_loss / len(train_loader)
    print(f"Epoch [{epoch}/{EPOCHS}] | Train Loss: {avg_loss:.4f}")

    # save model every epoch
    torch.save(
        model.state_dict(),
        f"/content/drive/MyDrive/brats_models/unet_epoch{epoch}.pth"
    )
    print("Model saved.")


Epoch [1/5] | Train Loss: 2.1532
Model saved.
Epoch [2/5] | Train Loss: 1.9245
Model saved.
Epoch [3/5] | Train Loss: 1.8117
Model saved.
Epoch [4/5] | Train Loss: 1.7843
Model saved.
Epoch [5/5] | Train Loss: 1.7154
Model saved.


In [24]:
import os

MODEL_SAVE_DIR = "/content/drive/MyDrive/brats_models"
print(os.listdir(MODEL_SAVE_DIR))


['unet_epoch2.pth', 'unet_epoch3.pth', 'unet_epoch4.pth', 'unet_epoch5.pth', 'unet_epoch1.pth']
